In [8]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout

from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import accuracy_score

In [9]:
BASE_PATH = "../Data"

isot_df = pd.read_csv(os.path.join(BASE_PATH, "ISOT", "isot_cleaned.csv"))
welfake_df = pd.read_csv(os.path.join(BASE_PATH, "WELFake", "welfake_cleaned.csv"))

# Safety fix
isot_df["content"] = isot_df["content"].astype(str).fillna("")
welfake_df["content"] = welfake_df["content"].astype(str).fillna("")

print("ISOT:", isot_df.shape)
print("WELFake:", welfake_df.shape)

ISOT: (38827, 2)
WELFake: (62749, 2)


In [10]:
# ISOT
X_train_isot, X_test_isot, y_train_isot, y_test_isot = train_test_split(
    isot_df["content"], isot_df["label"], test_size=0.2, random_state=42
)

# WELFake
X_train_wel, X_test_wel, y_train_wel, y_test_wel = train_test_split(
    welfake_df["content"], welfake_df["label"], test_size=0.2, random_state=42
)

In [11]:
vocab_size = 20000
max_len = 500

tokenizer = Tokenizer(num_words=vocab_size)

# Fit on combined data
tokenizer.fit_on_texts(pd.concat([X_train_isot, X_train_wel]))

# Convert to sequences
X_train_isot_seq = tokenizer.texts_to_sequences(X_train_isot)
X_test_isot_seq = tokenizer.texts_to_sequences(X_test_isot)

X_train_wel_seq = tokenizer.texts_to_sequences(X_train_wel)
X_test_wel_seq = tokenizer.texts_to_sequences(X_test_wel)

# Padding
X_train_isot_pad = pad_sequences(X_train_isot_seq, maxlen=max_len)
X_test_isot_pad = pad_sequences(X_test_isot_seq, maxlen=max_len)

X_train_wel_pad = pad_sequences(X_train_wel_seq, maxlen=max_len)
X_test_wel_pad = pad_sequences(X_test_wel_seq, maxlen=max_len)

In [17]:
def create_cnn_bilstm():
    model = Sequential([
        Embedding(input_dim=20000, output_dim=200, input_length=500),

        # CNN part
        Conv1D(128, 5, activation='relu'),
        MaxPooling1D(2),

        # BiLSTM part
        Bidirectional(LSTM(64, return_sequences=False)),

        Dense(32, activation='relu'),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(1024, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model

In [18]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [19]:
model_isot = create_cnn_bilstm()

history_isot = model_isot.fit(
    X_train_isot_pad, y_train_isot,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/10
437/437 ━━━━━━━━━━━━━━━━━━━━ 77s 169ms/step - accuracy: 0.9258 - loss: 0.1808 - val_accuracy: 0.9820 - val_loss: 0.0935
Epoch 2/10
437/437 ━━━━━━━━━━━━━━━━━━━━ 73s 167ms/step - accuracy: 0.9888 - loss: 0.0454 - val_accuracy: 0.9887 - val_loss: 0.0355
Epoch 3/10
437/437 ━━━━━━━━━━━━━━━━━━━━ 72s 164ms/step - accuracy: 0.9942 - loss: 0.0238 - val_accuracy: 0.9878 - val_loss: 0.0483
Epoch 4/10
437/437 ━━━━━━━━━━━━━━━━━━━━ 63s 145ms/step - accuracy: 0.9961 - loss: 0.0149 - val_accuracy: 0.9868 - val_loss: 0.0378


In [20]:
y_pred_isot = (model_isot.predict(X_test_isot_pad) > 0.5).astype("int32")

isot_acc = accuracy_score(y_test_isot, y_pred_isot)

print("ISOT Accuracy (CNN+BiLSTM):", isot_acc)

243/243 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step
ISOT Accuracy (CNN+BiLSTM): 0.9884110224053567


In [21]:
model_wel = create_cnn_bilstm()

history_wel = model_wel.fit(
    X_train_wel_pad, y_train_wel,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/10


C:\Users\Krish Kasnia\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


706/706 ━━━━━━━━━━━━━━━━━━━━ 119s 165ms/step - accuracy: 0.9106 - loss: 0.2383 - val_accuracy: 0.9293 - val_loss: 0.2476
Epoch 2/10
706/706 ━━━━━━━━━━━━━━━━━━━━ 117s 165ms/step - accuracy: 0.9630 - loss: 0.1119 - val_accuracy: 0.9550 - val_loss: 0.1514


In [22]:
y_pred_wel = (model_wel.predict(X_test_wel_pad) > 0.5).astype("int32")

wel_acc = accuracy_score(y_test_wel, y_pred_wel)

print("WELFake Accuracy (CNN+BiLSTM):", wel_acc)

393/393 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step
WELFake Accuracy (CNN+BiLSTM): 0.9250996015936255


In [23]:
hybrid_results = pd.DataFrame({
    "Dataset": ["ISOT", "WELFake"],
    "Model": ["CNN+BiLSTM", "CNN+BiLSTM"],
    "Accuracy": [isot_acc, wel_acc]
})

hybrid_results["Accuracy (%)"] = (hybrid_results["Accuracy"] * 100).round(2)

print(hybrid_results)

   Dataset       Model  Accuracy  Accuracy (%)
0     ISOT  CNN+BiLSTM  0.988411         98.84
1  WELFake  CNN+BiLSTM  0.925100         92.51
